# PROCESO DE EXTRACCIÓN, TRANSFORMACIÓN Y CARGA DE DATOS SOBRE MUNICIPIOS Y PROVINCIAS ESPAÑOLAS (ETL)

### Configuración del entorno (Local vs Google Colab)
Monta Google Drive e instala dependencias si se ejecuta en Google Colab; en local utiliza el directorio actual.


In [68]:
import os, sys
from pathlib import Path

# Detección de entorno: Local vs Google Colab
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    
    # Rutas habituales del proyecto en Drive (ajustar si tu carpeta tiene otro nombre)
    for ruta in [Path("/content/drive/MyDrive/TUI AI DASHBOARD CODE"),
                 Path("/content/drive/MyDrive/TFM - TUI/TUI AI DASHBOARD CODE"),
                 Path("/content/drive/MyDrive/TUI-AI-DASHBOARD-TFM"),
                 Path("/content/TUI-AI-DASHBOARD-TFM")]:
        if ruta.exists():
            os.chdir(ruta)
            break
            
    os.system("pip install -q duckdb geopandas pyarrow")

# Añadir la raíz y el Extractor a sys.path para importaciones directas
sys.path.extend([os.getcwd(), os.path.join(os.getcwd(), "Extractor")])


## 0.1 Carga de librerías esenciales para el proceso


In [69]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import json

#Libreria propia para este proyecto
import Formulas.FuncionesPropias as propias

## 1. Extracción de datos a través de APIs oficiales y públicas

En primer lugar, ejecutamos el main.py de extracción de datos, para actualizar mantener los datos actualizados.

Nota: El alumno recomienda no actualizarlo si ya se poseen los datos, ya que puede suponer un tiempo de espera elevado

In [70]:

actualizar = input("""¿Deseas actualizar los archivos de datos? (s/n)
         Ten en cuenta que puede tardar bastante en hacer las consultas, 
         por lo que si ya has cargado los datos una vez recomiendo no volver a hacerlo: """).strip().lower()

if actualizar=="s":
    try:
        %run Extractor/main.py #Ejecuta el archivo. Si falla continua con el cuaderno
    except Exception as e:
        print("Error al actualizar los datos")
else:
    print("No se han actualizado los archivos de datos.")

No se han actualizado los archivos de datos.


En la carga inicial de datos, es importante recalcar que municipiosDF y provinciasDF son extraidos manualmente del Instituto Geográfico Nacional, que no posee una API abierta al público. Se comentará más en el apartado 2.1 de este Notebook

In [71]:
# Información de flujos de movimiento
INE_puntosTur = pd.read_csv('Extractor/data/processed/flujo_ine_localidad.csv', sep=',', encoding='UTF-8')
f_INE_provincias = pd.read_csv('Extractor/data/processed/flujo_ine_provincia.csv', sep=',', encoding='UTF-8')
INE_provincias = pd.read_csv('Extractor/data/processed/INE_provincias.csv', sep=',', encoding='UTF-8')
municipios_data = pd.read_csv('Extractor/data/processed/municipios_espana.csv', sep=',', encoding='UTF-8')

# Información económica hotelera
rentabilidad_H = pd.read_csv('Extractor/data/processed/rentabilidad_hotelera.csv', sep=",", encoding= 'UTF_8')

#Información demográfica y geográfica de provincias y municipios

municipiosDF = pd.read_csv('Extractor/data/raw/MUNICIPIOS.csv', sep=';', encoding='latin1')
provinciasDF = pd.read_csv('Extractor/data/raw/PROVINCIAS.csv', sep=';', encoding='latin1')

# Métricas avanzadas

metricas_provincia = pd.read_csv('Extractor/data/processed/metricas_derivadas_actualizado.csv', sep=",", encoding= 'UTF_8')

## 2. Transformación de la información extraida

### 2.0 Datos ya preparados: Métricas avanzadas

El dataframe de metricas ya viene listo para ser utilizado, no necesita ni le falta ninguna información

In [72]:
metricas_provincia.describe().round(2)


,cod_prov,poblacion,total_viajeros,media_mensual_viajeros,std_mensual_viajeros,presion_turistica,indice_estacionalidad,n_meses
count,52.00,52.00,52.00,52.00,52.00,52.00,52.00,52.0
mean,26.50,911252.06,2303221.73,191935.14,58391.28,2.41,0.29,12.0
std,15.15,1202328.15,3217046.77,268087.23,123480.25,1.67,0.16,0.0
min,1.00,83517.00,62844.00,5237.00,1004.77,0.75,0.06,12.0
25%,13.75,324458.75,538149.25,44845.77,9035.54,1.43,0.18,12.0
50%,26.50,607127.00,1088766.50,90730.54,24624.87,2.03,0.25,12.0
75%,39.25,1019945.25,2496757.00,208063.08,57630.71,2.72,0.33,12.0
max,52.00,6751251.00,14282218.00,1190184.83,855499.74,10.84,0.81,12.0


### 2.1 Datos geográficos de municipios y provincias

In [73]:
# Primero revisamos la información y nombres de columnas en nuestros archivos descargados directamente del CNIG
provinciasDF.info()
print('-------------------------------------------------------')
municipiosDF.info()

municipiosDF.head(1)

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   COD_PROV            52 non-null     int64
 1   PROVINCIA           52 non-null     str  
 2   COD_CA              52 non-null     int64
 3   COMUNIDAD_AUTONOMA  52 non-null     str  
 4   CAPITAL             52 non-null     str  
dtypes: int64(2), str(3)
memory usage: 3.8 KB
-------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 8132 entries, 0 to 8131
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   COD_INE                   8132 non-null   int64
 1   ID_REL                    8132 non-null   int64
 2   COD_GEO                   8132 non-null   int64
 3   COD_PROV                  8132 non-null   int64
 4   PROVINCIA                 8132 non-null   str  
 5   NOMBRE

,COD_INE,ID_REL,COD_GEO,COD_PROV,PROVINCIA,NOMBRE_ACTUAL,POBLACION_MUNI,SUPERFICIE,PERIMETRO,COD_INE_CAPITAL,CAPITAL,POBLACION_CAPITAL,HOJA_MTN25,LONGITUD_ETRS89_REGCAN95,LATITUD_ETRS89_REGCAN95,ORIGENCOOR,ALTITUD,ORIGENALTITUD
0,1001000000,1010014,1010,1,Araba/Álava,Alegría-Dulantzi,2961,"1994,5872",35069,1001000101,Alegría-Dulantzi,2842,0113-3,"-2,512507724","42,84045247",Detección automática,568,MDT


Como la información de estos dos Dataframes está de por si bastante bien estructurada, no necesita ningún trabajo de transformación, por lo que procedemos a incluir en municipiosDF la información de su provincia para en futuros apartados crear ratios con esta información

In [74]:
#En primer lugar, vamos a combinar los dos DFs extraidos manualmente del centro de descargas del IGN 
# https://centrodedescargas.cnig.es/CentroDescargas/nomenclator-geografico-municipios-entidades-poblacion

# Merge para unir información de provincia por su codigo
municipiosDF = municipiosDF.merge(provinciasDF[['COD_PROV','COD_CA', 'COMUNIDAD_AUTONOMA']], on='COD_PROV')

municipiosDF = municipiosDF.drop(['ID_REL', 'HOJA_MTN25', 'ORIGENCOOR', 'ORIGENALTITUD'], axis=1)

municipiosDF.head()
municipiosDF.info()


<class 'pandas.DataFrame'>
RangeIndex: 8132 entries, 0 to 8131
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   COD_INE                   8132 non-null   int64
 1   COD_GEO                   8132 non-null   int64
 2   COD_PROV                  8132 non-null   int64
 3   PROVINCIA                 8132 non-null   str  
 4   NOMBRE_ACTUAL             8132 non-null   str  
 5   POBLACION_MUNI            8132 non-null   int64
 6   SUPERFICIE                8132 non-null   str  
 7   PERIMETRO                 8132 non-null   int64
 8   COD_INE_CAPITAL           8132 non-null   int64
 9   CAPITAL                   8132 non-null   str  
 10  POBLACION_CAPITAL         8132 non-null   int64
 11  LONGITUD_ETRS89_REGCAN95  8132 non-null   str  
 12  LATITUD_ETRS89_REGCAN95   8132 non-null   str  
 13  ALTITUD                   8132 non-null   str  
 14  COD_CA                    8132 non-null   int64
 15

### 2.2 Datos de caracter turístico extraidos mediante la API del Instituto Nacional de Estadística (INE)

Comenzamos con una revisión general de los datos. 
Como la API nos devuelve una estructura similar para todos estos Dataframes, el proceso va a ser similar, pero con casos especiales para cada situación.

In [75]:
#INE_puntosTur.head()
#INE_puntosTur.shape --> Resultado: (6610, 11)
INE_puntosTur.info()

<class 'pandas.DataFrame'>
RangeIndex: 13220 entries, 0 to 13219
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   COD                     13220 non-null  str    
 1   Nombre                  13220 non-null  str    
 2   T3_Unidad               13220 non-null  str    
 3   T3_Escala               13220 non-null  str    
 4   Fecha                   13220 non-null  str    
 5   T3_Periodo              13220 non-null  str    
 6   T3_TipoDato             13220 non-null  str    
 7   Anyo                    13220 non-null  int64  
 8   Valor                   11380 non-null  float64
 9   MetaData_json           13220 non-null  str    
 10  tabla_id                13220 non-null  int64  
 11  _meta.fecha_extraccion  13220 non-null  str    
dtypes: float64(1), int64(2), str(9)
memory usage: 9.0 MB


In [76]:
INE_provincias.head()
#INE_provincias[INE_provincias['INE_EOH_PROV.indicador']=='Viajero'].head()
#INE_provincias.info()

,COD,Nombre,T3_Unidad,T3_Escala,Fecha,T3_Periodo,T3_TipoDato,Anyo,Valor,MetaData_json,tabla_id,_meta.fecha_extraccion
0,DPOP1,Total Nacional. Total. Total habitantes. Perso...,Personas,,2021-01-01T00:00:00.000+01:00,A,Definitivo,2021,47385107.0,"[{""Id"": 16473, ""T3_Variable"": ""Comunidades y C...",2852,2026-09-14T13:34:49.147336
1,DPOP2,Total Nacional. Hombres. Total habitantes. Per...,Personas,,2021-01-01T00:00:00.000+01:00,A,Definitivo,2021,23222953.0,"[{""Id"": 16473, ""T3_Variable"": ""Comunidades y C...",2852,2026-09-14T13:34:49.147336
2,DPOP3,Total Nacional. Mujeres. Total habitantes. Per...,Personas,,2021-01-01T00:00:00.000+01:00,A,Definitivo,2021,24162154.0,"[{""Id"": 16473, ""T3_Variable"": ""Comunidades y C...",2852,2026-09-14T13:34:49.147336
3,DPOP160,Albacete. Total. Total habitantes. Personas.,Personas,,2021-01-01T00:00:00.000+01:00,A,Definitivo,2021,386464.0,"[{""Id"": 3, ""T3_Variable"": ""Provincias"", ""Nombr...",2852,2026-09-14T13:34:49.147336
4,DPOP161,Albacete. Hombres. Total habitantes. Personas.,Personas,,2021-01-01T00:00:00.000+01:00,A,Definitivo,2021,193205.0,"[{""Id"": 3, ""T3_Variable"": ""Provincias"", ""Nombr...",2852,2026-09-14T13:34:49.147336


Cambios necesarios:
- INE_localidades y localidades_data se deben combinar
- En f_INE_provincias, INE_Provincias, Provincias_data y OpenStreetDF se deben combinar

El resultado debe ser dos DataFrames, uno para localidades, y otro para provincias. Posteriormente se deberá hacer un Left Join en las localidades para los municipios. Posteriormente, se deben combinar ambos Dfs

Comenzamos filtrando y reduciendo las columnas en localidades_data

In [77]:
municipios_data.nunique()

COD                       24396
Nombre                    24345
T3_Unidad                     1
T3_Escala                     1
Fecha                         1
T3_Periodo                    1
T3_TipoDato                   1
Anyo                          1
Valor                      6179
MetaData_json             24396
tabla_id                      1
_meta.fecha_extraccion        1
dtype: int64

In [78]:
#Revisamos el interior de los json que encontraremos en gran parte de los datasets trabajados
pd.set_option('display.max_colwidth', None) # Comando para poder ver cadenas largas de texto
print(municipios_data['MetaData_json'].tail())
      
pd.reset_option('display.max_colwidth') # Retrocedemos el comando

24391     [{"Id": 6270, "T3_Variable": "Municipios", "Nombre": "Zúñiga", "Codigo": "31265"}, {"Id": 452, "T3_Variable": "Sexo", "Nombre": "Hombres", "Codigo": "1"}, {"Id": 8677, "T3_Variable": "Tamaño de los municipios", "Nombre": "Total habitantes", "Codigo": "0"}, {"Id": 20258, "T3_Variable": "Tipo de dato", "Nombre": "Personas", "Codigo": ""}]
24392     [{"Id": 6270, "T3_Variable": "Municipios", "Nombre": "Zúñiga", "Codigo": "31265"}, {"Id": 453, "T3_Variable": "Sexo", "Nombre": "Mujeres", "Codigo": "2"}, {"Id": 8677, "T3_Variable": "Tamaño de los municipios", "Nombre": "Total habitantes", "Codigo": "0"}, {"Id": 20258, "T3_Variable": "Tipo de dato", "Nombre": "Personas", "Codigo": ""}]
24393      [{"Id": 5824, "T3_Variable": "Municipios", "Nombre": "Zurgena", "Codigo": "04103"}, {"Id": 451, "T3_Variable": "Sexo", "Nombre": "Total", "Codigo": "0"}, {"Id": 8677, "T3_Variable": "Tamaño de los municipios", "Nombre": "Total habitantes", "Codigo": "0"}, {"Id": 20258, "T3_Variable": "Tipo 

Observamos que el Código identificador de INE se encuentra en una columna de multiples Json en la columna MetaData_json. Lo mejor será definir una funcion para extraer esta información adecuadamente

In [79]:
# Primero, dividimos la columna de 'Nombre' usando el metodo .str.split()
columnas = ['Localidad', 'Genero', 'Metrica', 'Unidad']
municipios_data[columnas] = municipios_data['Nombre'].str.split('.', n=3, expand=True)

for col in columnas:
    municipios_data[col] = municipios_data[col].str.strip() #Eliminamos los espacios al inicio y final con un bucle

# A continuación, extraemos el codigo INE del json en la tabla
municipios_data['COD_INE'] = municipios_data['MetaData_json'].apply(propias.obtener_codigo_INE)

# Suprimimos las columnas que no vamos a necesitar
municipios_data.drop(columns=['COD', 'Nombre', 'Fecha', 'T3_Unidad', 'T3_Escala', 'T3_Periodo', 'T3_TipoDato','Anyo', 'tabla_id', 'MetaData_json', '_meta.fecha_extraccion'],
                     inplace=True
                     )

# Como nos interesa la poblacion total por municipio, filtramos Genero == 'Total
municipios_data = municipios_data[municipios_data['Genero'] == 'Total']

# Comprobamos el resultado
print(municipios_data.head())

print('--------------------------------------------------------------------------------------------------------')
print('Nota: realmente solo nos interesa COD_INE, Valor y localidad, pero el resto de columnas aportan contexto')

     Valor Localidad Genero           Metrica     Unidad COD_INE
0     73.0    Ababuj  Total  Total habitantes  Personas.   44001
3    849.0    Abades  Total  Total habitantes  Personas.   40001
6    337.0    Abadía  Total  Total habitantes  Personas.   10001
9   2239.0    Abadín  Total  Total habitantes  Personas.   27001
12  7768.0   Abadiño  Total  Total habitantes  Personas.   48001
--------------------------------------------------------------------------------------------------------
Nota: realmente solo nos interesa COD_INE, Valor y localidad, pero el resto de columnas aportan contexto


In [80]:
# Comprobamos si hemos extraido bien COD_INE
print( """COMPROBACIÓN DE municipios_data
      """)
print(f"Valores nulos: {municipios_data['COD_INE'].isna().sum()}")
print('--------------------------------')
print(f"Valores duplicados (esperados {municipios_data['COD_INE'].count()}): {municipios_data['COD_INE'].nunique()}")

COMPROBACIÓN DE municipios_data
      
Valores nulos: 0
--------------------------------
Valores duplicados (esperados 8132): 8132


A continuación, adaptaremos INE_localidades.

En la columna T3_Unidad tenemos dos valores, Viajeros y pernoctaciones, por lo que nos interesa quedarnos con ambos pero en distintas columnas.

In [81]:
INE_puntosTur.head(2) #2 para que ocupe poco espacio en la salida

,COD,Nombre,T3_Unidad,T3_Escala,Fecha,T3_Periodo,T3_TipoDato,Anyo,Valor,MetaData_json,tabla_id,_meta.fecha_extraccion
0,EOT2611,Nacional. Viajeros. Vitoria-Gastéiz. Residente...,Viajeros,,2026-07-01T00:00:00.000+02:00,M07,Provisional,2026,18734.0,"[{""Id"": 284332, ""T3_Variable"": ""Concepto turís...",2078,2026-09-14T13:35:04.631537
1,EOT2611,Nacional. Viajeros. Vitoria-Gastéiz. Residente...,Viajeros,,2026-06-01T00:00:00.000+02:00,M06,Provisional,2026,19464.0,"[{""Id"": 284332, ""T3_Variable"": ""Concepto turís...",2078,2026-09-14T13:35:04.631537


In [82]:
#Como el DF tiene dos niveles en el mismo origen de datos, vamos a partirlo en dos y despues unir sus columnas resultantes.

#El primer nivel será en INE_puntosTur2
INE_puntosTur2 = INE_puntosTur[(~INE_puntosTur['Nombre'].str.startswith('Nacional'))]
columnas = ['localidad', 'metrica', 'Campos', 'Origen', 'Tipo']

# Split de las columnas iniciales
INE_puntosTur2[columnas] = INE_puntosTur2['Nombre'].str.split('.', n=4, expand=True)

# Extraemos COD_INE del json
INE_puntosTur2['COD_INE'] = INE_puntosTur2['MetaData_json'].apply(propias.obtener_codigo_INE)

# Borramos columnas que no necesitamos (aunque despues pivotaremos, por lo que no es estrictamente necesario, mejora los siguientes pasos)
INE_puntosTur2.drop(columns=['Nombre', 'Fecha', 'T3_Escala', 'T3_TipoDato', 'tabla_id', 'MetaData_json', '_meta.fecha_extraccion', 'Tipo'],
                     inplace=True
                     )

# Exploración de distintas métricas
print('Valores únicos en cada columna:')
print(INE_puntosTur2.nunique()) 
print('-----------------------------')
print(INE_puntosTur2.Anyo.value_counts())
print('-----------------------------')
print(INE_puntosTur2.metrica.value_counts())
print('-----------------------------')
print(INE_puntosTur2.Origen.value_counts())
print('-----------------------------')
# Conclusiones:
 # Solo nos interesa mantener viajeros y pernoctaciones (Total categorias es el total)
 # Nos interesa diferenciar el origen en dos columnas, pero cambiando los nombres


INE_puntosTur2 = (INE_puntosTur2[(INE_puntosTur2['metrica'] != ' Total categorías')]
                    .rename(columns= {'T3_Periodo': 'periodo'}) # Renombramos la columna para su posterior uso
                    )

INE_puntosTur2['Origen'] = (INE_puntosTur2['Origen']
                              .str.strip() #Para limpiar el texto, ya que .replace solo no daba resultado
                              .replace({'Residentes en España': 'Nacional',
                                        'Residentes en el Extranjero': 'Extranjero'}))


INE_puntosTur2 =(INE_puntosTur2.sort_values('Valor', ascending=False)
 .pivot_table(index= ['COD_INE', 'localidad'],
                             columns= ['metrica', 'Origen'],
                             values= ['Valor', 'periodo'],
                             aggfunc= {'Valor': 'sum', 'periodo': 'first'})
 .reset_index())

INE_puntosTur2.columns = INE_puntosTur2.columns.map(' '.join) #join con valor vacio porque ya existe un valor vacio en las metricas

print('Resultado del Dataframe')
INE_puntosTur2.head(3)


Valores únicos en cada columna:
COD            164
T3_Unidad        2
T3_Periodo      12
Anyo             3
Valor         2804
localidad       40
metrica          3
Campos           4
Origen           2
COD_INE         41
dtype: int64
-----------------------------
Anyo
2025    1954
2026    1142
2024     814
Name: count, dtype: int64
-----------------------------
metrica
Viajero             1811
Pernoctaciones      1811
Total categorías     288
Name: count, dtype: int64
-----------------------------
Origen
Residentes en el Extranjero    1956
Residentes en España           1954
Name: count, dtype: int64
-----------------------------
Resultado del Dataframe


,COD_INE,localidad,Valor Pernoctaciones Extranjero,Valor Pernoctaciones Nacional,Valor Viajero Extranjero,Valor Viajero Nacional,periodo Pernoctaciones Extranjero,periodo Pernoctaciones Nacional,periodo Viajero Extranjero,periodo Viajero Nacional
0,03018,Altea,365608.0,291281.0,109312.0,136312.0,M09,M08,M05,M08
1,04032,Carboneras,9895.0,109766.0,4273.0,46406.0,M09,M08,M09,M07
2,04902,"Ejido, El",25776.0,52671.0,7614.0,18840.0,M08,M08,M08,M08


In [83]:
#El segundo nivel mencionado anteriormente será en INE_puntosTur_nac, con los valores de residentes en españa y extranjeros
# Ahora, repetimos con la otra parte del dataframe
INE_puntosTur_nac = INE_puntosTur[INE_puntosTur['Nombre'].str.startswith('Nacional')]

# Misma operación con la otra metrica, pero diferencia en las columnas que desagregamos
columnas= ['tipo', 'metrica', 'localidad', 'tipo residente']

INE_puntosTur_nac[columnas] = INE_puntosTur_nac['Nombre'].str.split('.', n=3, expand=True)

# Funcion de MetaData_json
INE_puntosTur_nac['COD_INE'] = INE_puntosTur_nac['MetaData_json'].apply(propias.obtener_codigo_INE)

INE_puntosTur_nac.drop(columns=['Nombre', 'Fecha', 'T3_Escala', 'T3_TipoDato','tipo', 'tabla_id', 'MetaData_json', '_meta.fecha_extraccion'],
                     inplace=True
                        )

# Revisamos un poco los valores que nos encontramos
print(INE_puntosTur_nac.nunique()) 
print('-----------------------------')
print(INE_puntosTur_nac.metrica.value_counts()) #Otra vez, elegimos 2026
print('-----------------------------')
print(INE_puntosTur_nac['tipo residente'].value_counts()) #Genera un problema, ya que algunas filas no presentan bien si es español o extranjero

#Primero, solucionaremos el problema de tipo residente, apoyandonos en numpy

condiciones = [INE_puntosTur_nac['tipo residente'].str.contains('España', case=False, na = False),
               INE_puntosTur_nac['tipo residente'].str.contains('extranjero', case=False, na = False)
               ]
eleccion = ['Nacional', 'Extranjero']

INE_puntosTur_nac['tipo turista'] = np.select(condiciones, eleccion, default= ' ')

# Modificamos el DF
INE_puntosTur_nac = (INE_puntosTur_nac[(INE_puntosTur_nac['Anyo'] == 2026) & #Nos quedamos con 2026 porque tiene más metricas
                                           (INE_puntosTur_nac['metrica'] != ' Establecimientos hoteleros')
                                           ] 
                       .rename(columns={'T3_Periodo': 'periodo'}) #Aprovechamos para renombrar esta columna
                       )                 

# metodo rapido para modificar valor, ya que será necesario en el siguiente
INE_puntosTur_nac.loc[INE_puntosTur_nac['metrica'].str.strip() == 'Viajeros', 'metrica'] = 'Viajero'

# Para lograr tener el mes de mayor valor en la pivotacion, primero ordenaremos para poder mantener el mes con mayor numero de viajeros
INE_puntosTur_nac = (INE_puntosTur_nac
                       .sort_values('Valor', ascending=False) #Ordenamos
                       .pivot_table(index=['COD_INE','localidad'],
                                    columns= ['metrica', 'tipo turista'],
                                    values= ['Valor','periodo'],
                                    aggfunc= {'Valor':'sum', 'periodo':'first'}) #Pivotamos
                       .reset_index()
                       )

# Eliminamos bandas y unificamos en el nombre de columna
INE_puntosTur_nac.columns = INE_puntosTur_nac.columns.map(' '.join) #join con valor vacio porque ya existe un valor vacio en las metricas

print('-----------------------------')
print('Resultado del Dataframe:')
INE_puntosTur_nac.head(3)


COD                388
T3_Unidad            2
T3_Periodo          12
Anyo                 3
Valor             7931
metrica              3
localidad           74
tipo residente      52
COD_INE             97
dtype: int64
-----------------------------
metrica
Viajeros                      3456
Pernoctaciones                3456
Establecimientos hoteleros    2398
Name: count, dtype: int64
-----------------------------
tipo residente
Residentes en España.                                             3456
Residentes en el extranjero.                                      3456
03063-Denia. Residentes en España.                                  48
03063-Denia. Residentes en el extranjero.                           48
04066-Níjar. Residentes en España.                                  48
04066-Níjar. Residentes en el extranjero.                           48
07014-Capdepera. Residentes en España.                              48
07014-Capdepera. Residentes en el extranjero.                       4

,COD_INE,localidad,Valor Pernoctaciones Extranjero,Valor Pernoctaciones Nacional,Valor Viajero Extranjero,Valor Viajero Nacional,periodo Pernoctaciones Extranjero,periodo Pernoctaciones Nacional,periodo Viajero Extranjero,periodo Viajero Nacional
0,01059,Vitoria-Gastéiz,175065.0,293398.0,57422.0,136922.0,M07,M04,M07,M04
1,03014,Alicante,1063874.0,367537.0,360259.0,181608.0,M07,M06,M05,M06
2,03031,Benidorm,4105484.0,2238270.0,834735.0,551668.0,M07,M07,M05,M07


Antes de concatenar (anexar) de vuelta los dos dataframes, debemos solucionar la diferencia entre las columnas resultado las operaciones paralelas.

Hay que tener en cuenta que algunos códigos INE son un valor alfanumérico origen de la encuesta EOH. debemos transformar este código al codigo INE de la provincia, o en su defecto realizar una combinación de columnas a partir del nombre

In [84]:
#Unificados nombres de columnas
columnas = (INE_puntosTur_nac.columns
            .str.strip() #Eliminamos espacios al inicio y final
            .str.replace('  ', ' ', regex= False) # suprimimos dobles espaciados
            .str.upper() #Normalizamos todas las columnas a mayusculas
            )

# Aplicamos
INE_puntosTur2.columns = columnas
INE_puntosTur_nac.columns = columnas

# Unimos los Dfs
INE_puntosTur = pd.concat([INE_puntosTur2, INE_puntosTur_nac], ignore_index=True)

# Filtramos para suprimir que no tienen valores
INE_puntosTur = INE_puntosTur[INE_puntosTur[INE_puntosTur.columns[2]] > 0]


# Revisamos
INE_puntosTur.sort_values('LOCALIDAD')


,COD_INE,LOCALIDAD,VALOR PERNOCTACIONES EXTRANJERO,VALOR PERNOCTACIONES NACIONAL,VALOR VIAJERO EXTRANJERO,VALOR VIAJERO NACIONAL,PERIODO PERNOCTACIONES EXTRANJERO,PERIODO PERNOCTACIONES NACIONAL,PERIODO VIAJERO EXTRANJERO,PERIODO VIAJERO NACIONAL
47,A1,Adeje,6069924.0,302222.0,859460.0,76113.0,M07,M07,M03,M07
48,A5,Albacete,32020.0,171794.0,16601.0,111280.0,M02,M05,M04,M05
49,A6,Albarracín,6936.0,37844.0,4255.0,20366.0,M05,M04,M03,M04
50,A8,Algeciras,73041.0,88538.0,43367.0,43260.0,M07,M03,M03,M03
39,03014,Alicante,1063874.0,367537.0,360259.0,181608.0,M07,M06,M05,M06
...,...,...,...,...,...,...,...,...,...,...
12,07061,Sóller,1005730.0,39952.0,265620.0,16868.0,M08,M06,M09,M06
31,K7,Teguise,4480139.0,681309.0,605654.0,134795.0,M08,M08,M03,M08
24,25043,"Vall de Boí, La",12562.0,217634.0,5576.0,75235.0,M08,M08,M07,M08
32,M4,Yaiza,9065358.0,1276782.0,1222484.0,236113.0,M10,M08,M10,M08


Pasando a los datos agregados por provincias, el flujo de transformaciones será similar a los dfs de localidades, con pequeñas variaciones

In [85]:
#Revisamos el interior de los json que encontraremos en gran parte de los datasets trabajados
pd.set_option('display.max_colwidth', None) # Comando para poder ver cadenas largas de texto
print(f_INE_provincias['MetaData_json'].tail())
      
pd.reset_option('display.max_colwidth') # Retrocedemos el comando

10075    [{"Id": 8995, "T3_Variable": "Comunidades y Ciudades Autónomas", "Nombre": "Melilla", "Codigo": "19"}, {"Id": 284333, "T3_Variable": "Concepto turístico", "Nombre": "Pernoctaciones", "Codigo": "C"}, {"Id": 9834, "T3_Variable": "TIPO DE CATEGORIA", "Nombre": "Total categorías", "Codigo": ""}, {"Id": 19965, "T3_Variable": "RESIDENCIA/ORIGEN", "Nombre": "Residentes en el Extranjero", "Codigo": ""}, {"Id": 72, "T3_Variable": "Tipo de dato", "Nombre": "Dato", "Codigo": "0"}]
10076    [{"Id": 8995, "T3_Variable": "Comunidades y Ciudades Autónomas", "Nombre": "Melilla", "Codigo": "19"}, {"Id": 284333, "T3_Variable": "Concepto turístico", "Nombre": "Pernoctaciones", "Codigo": "C"}, {"Id": 9834, "T3_Variable": "TIPO DE CATEGORIA", "Nombre": "Total categorías", "Codigo": ""}, {"Id": 19965, "T3_Variable": "RESIDENCIA/ORIGEN", "Nombre": "Residentes en el Extranjero", "Codigo": ""}, {"Id": 72, "T3_Variable": "Tipo de dato", "Nombre": "Dato", "Codigo": "0"}]
10077    [{"Id": 8995, "T3_Varia

In [86]:
# Comenzamos revisando los
print(f_INE_provincias.nunique())
print('-----------------------------')
print(f_INE_provincias['T3_Unidad'].value_counts())
print('-----------------------------')
print(f_INE_provincias.Nombre.value_counts())

#NOTA: en este DF están mezcladas provincias, comunidades y total nacional, conviene quedarse solo con provincias
# Para quedarnos con Provincias, se debe filtrar en la columna MetaData_json, que contiene el valor crudo en formato json, aunque no es necesario desagregarlo

f_INE_provincias = f_INE_provincias[f_INE_provincias['MetaData_json'].str.contains('Provincia', case=False, na=False)]

columnas=['Provincia', 'metrica', 'Origen']

f_INE_provincias[columnas] = f_INE_provincias['Nombre'].str.split('.', n=2, expand=True)

# Extraemos COD_PROV del json, valor numerico que define la provincia
f_INE_provincias['COD_PROV'] = f_INE_provincias['MetaData_json'].apply(propias.obtener_codigo_provincia_INE)

f_INE_provincias = f_INE_provincias.drop(columns= ['Nombre', 'T3_Escala', 'Fecha', 'T3_TipoDato', 'MetaData_json', 'tabla_id', '_meta.fecha_extraccion'])


#Filtramos años. En este caso nos quedamos con 2025 ya que contiene los doce meses
f_INE_provincias = (f_INE_provincias[f_INE_provincias['Anyo'] == 2025]
                     .rename(columns={'T3_Periodo': 'periodo'}))
                     
f_INE_provincias=f_INE_provincias.sort_values('Valor', ascending=False)


f_INE_provincias.loc[f_INE_provincias['metrica'].str.strip() == 'Viajeros', 'metrica'] = 'Viajero'
f_INE_provincias['metrica'] = f_INE_provincias['metrica'].str.strip()

f_INE_provincias = f_INE_provincias.pivot_table(index=['COD_PROV', 'Provincia'],
                                    columns= ['metrica'],
                                    values= ['Valor','periodo'],
                                    aggfunc= {'Valor':'sum', 'periodo':'first'})

f_INE_provincias.columns = f_INE_provincias.columns.map(' '.join)
f_INE_provincias.columns = (f_INE_provincias.columns
            .str.strip() #Eliminamos espacios al inicio y final
            .str.replace('  ', ' ', regex= False) # suprimimos dobles espaciados
            .str.upper() #Normalizamos todas las columnas a mayusculas
            )

f_INE_provincias = f_INE_provincias.reset_index()
f_INE_provincias.head()

COD                        420
Nombre                     420
T3_Unidad                    2
T3_Escala                    1
Fecha                       24
T3_Periodo                  12
T3_TipoDato                  2
Anyo                         3
Valor                     8933
MetaData_json              420
tabla_id                     1
_meta.fecha_extraccion       1
dtype: int64
-----------------------------
T3_Unidad
Viajeros          5040
Pernoctaciones    5040
Name: count, dtype: int64
-----------------------------
Nombre
Nacional. Viajeros. Total categorías. Total.                          24
Nacional. Viajeros. Total categorías. Residentes en España.           24
Nacional. Viajeros. Total categorías. Residentes en el extranjero.    24
Nacional. Pernoctaciones. Total categorías. Total.                    24
Nacional. Pernoctaciones. Total categorías. Residentes en España.     24
                                                                      ..
Melilla. Viajeros. Residente

,COD_PROV,Provincia,VALOR PERNOCTACIONES,VALOR VIAJERO,PERIODO PERNOCTACIONES,PERIODO VIAJERO
0,01,Alava,2208517.0,1027118.0,M08,M08
1,02,Albacete,1737927.0,842801.0,M09,M09
2,03,Alicante,37680371.0,10134012.0,M08,M08
3,04,Almería,10606166.0,3125996.0,M08,M08
4,05,Avila,1243810.0,781100.0,M08,M08


Para la rentabilidad hotelera, es importante conocer los dos indicadores que comparte este DataFrame
- **ADR (Average Daily Rate)**: Es el ingreso promedio por habitación ocupada (las habitaciones no alguiladas no cuentan)
- **RevPar (Revenue Per Available Room)**: Ingreso promedio por habitación disponible (ocupadas y disponibles).

In [87]:
#rentabilidad_H.info()

rentabilidad_H = rentabilidad_H.copy()

rentabilidad_H = rentabilidad_H[rentabilidad_H['MetaData_json'].str.contains('Provincias')]
columnas = ['Provincia', 'Metrica', 'Categoria', 'TipoDato']

rentabilidad_H[columnas] = rentabilidad_H['Nombre'].str.split('.', n=3, expand=True)

#Normalizamos nombres de columnas y posteriormente suprimimos las que no necesitamos
for col in columnas:
    rentabilidad_H[col] = (rentabilidad_H[col]
            .str.strip() #Eliminamos espacios al inicio y final
            .str.replace(r'\s+', ' ', regex= True) # suprimimos dobles espaciados
            .str.replace(r'[^\w\s]', '', regex= True)) # Truco para eliminar signos de puntuación

# Similar al anterior, usamos segunda formula para cod de provincia
rentabilidad_H['COD_INE'] = rentabilidad_H['MetaData_json'].apply(propias.obtener_codigo_provincia_INE)

rentabilidad_H = rentabilidad_H.drop(['Nombre', 'T3_TipoDato', 'T3_Escala', 'T3_Unidad', 'MetaData_json', 'tabla_id', '_meta.fecha_extraccion', 'Categoria'], axis=1)


#Primero, revisamos los datos que nos podemos encontrar
print(rentabilidad_H.nunique())

# Renombramos para acortar
rentabilidad_H['Metrica'] = rentabilidad_H['Metrica'].replace({'Ingresos por habitación disponible RevPAR': 'RevPar',
                                            'Tarifa media diaria ADR':'ADR'})

# Reemplazamos para acortar TipoDato
rentabilidad_H['TipoDato'] =rentabilidad_H['TipoDato'].replace({'Tasa de variación interanual': 'Tasa Var'})

# Ordenamos  por valor mas algo y renombramos Periodo, para la posterior pivotacion
rentabilidad_H = (rentabilidad_H
         .sort_values('Valor', ascending= False)
         .rename(columns={'T3_Periodo':'Periodo'})
         )


# Sacamos la media de ADR por provincia como un dato extra.
# Es importante remarcar que ADR es ya de por si un ingreso medio de habitaciones ocupadas
# Por lo tanto, hacer la media por provincia no desvirtua tanto el dato como la media de RevPar 
rentAnual=(rentabilidad_H
           .groupby(['Provincia', 'TipoDato', 'Metrica'], as_index=False)['Valor']
           .mean()
           )

# Filtramos los datos para quedarnos con el ADR medio de cada provincia
rentAnual = rentAnual[(rentAnual['Metrica'] == 'ADR') & (rentAnual['TipoDato'] == 'Dato') ]

rentAnual = rentAnual.rename(columns={'Valor':'ADR Anual Medio'})
rentAnual = rentAnual.drop(['TipoDato', 'Metrica'], axis= 1)

# Pivotamos tablas, como hemos ordenado por valores mas alto podemos mantener Valor y periodo en first
rentabilidad_H = rentabilidad_H.pivot_table(index=['COD_INE', 'Provincia'],
                          columns=['Metrica','TipoDato'],
                          values=['Valor', 'Periodo'],
                          aggfunc={'Valor':'first', 'Periodo':'first'},
                          ) #Como ADR y RevPar son indicadores, no tiene sentido sumarlos

# Unimos las bandas y columnas para tener un unico nivel de columnas
rentabilidad_H.columns = rentabilidad_H.columns.map(' '.join)

# Reseteamos indices (Provincia)
rentabilidad_H = rentabilidad_H.reset_index()

rentabilidad_H.head()

COD            208
Fecha           12
T3_Periodo      12
Anyo             2
Valor         2182
Provincia       52
Metrica          2
TipoDato         2
COD_INE         52
dtype: int64


,COD_INE,Provincia,Periodo ADR Dato,Periodo ADR Tasa de variacin interanual,Periodo Ingresos por habitacin disponible RevPAR Dato,Periodo Ingresos por habitacin disponible RevPAR Tasa de variacin interanual,Valor ADR Dato,Valor ADR Tasa de variacin interanual,Valor Ingresos por habitacin disponible RevPAR Dato,Valor Ingresos por habitacin disponible RevPAR Tasa de variacin interanual
0,01,Arabalava,M07,M06,M07,M10,117.12,13.91,89.31,27.88
1,02,Albacete,M09,M11,M09,M09,75.62,20.13,50.10,54.79
2,03,AlicanteAlacant,M08,M11,M08,M11,159.22,8.13,136.07,11.54
3,04,Almera,M08,M09,M08,M09,163.14,16.26,137.15,19.80
4,05,vila,M04,M06,M08,M03,75.19,6.75,36.13,30.83


In [88]:
# Unimos las dos tablas trabjadas en la celda anterior
rentabilidad_H = rentabilidad_H.merge(rentAnual, how= 'left', on= 'Provincia', suffixes=('', '_A'))

# Borramos rentAnual para ahorrar memoria
del rentAnual

#rentabilidad_H = rentabilidad_H.drop(['Periodo RevPar Tasa Var' ], axis= 1 )
rentabilidad_H.info()

rentabilidad_H.head()


<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 11 columns):
 #   Column                                                                        Non-Null Count  Dtype  
---  ------                                                                        --------------  -----  
 0   COD_INE                                                                       52 non-null     str    
 1   Provincia                                                                     52 non-null     str    
 2   Periodo ADR Dato                                                              52 non-null     str    
 3   Periodo ADR Tasa de variacin interanual                                       52 non-null     str    
 4   Periodo Ingresos por habitacin disponible RevPAR Dato                         52 non-null     str    
 5   Periodo Ingresos por habitacin disponible RevPAR Tasa de variacin interanual  52 non-null     str    
 6   Valor ADR Dato                                 

,COD_INE,Provincia,Periodo ADR Dato,Periodo ADR Tasa de variacin interanual,Periodo Ingresos por habitacin disponible RevPAR Dato,Periodo Ingresos por habitacin disponible RevPAR Tasa de variacin interanual,Valor ADR Dato,Valor ADR Tasa de variacin interanual,Valor Ingresos por habitacin disponible RevPAR Dato,Valor Ingresos por habitacin disponible RevPAR Tasa de variacin interanual,ADR Anual Medio
0,01,Arabalava,M07,M06,M07,M10,117.12,13.91,89.31,27.88,92.070833
1,02,Albacete,M09,M11,M09,M09,75.62,20.13,50.10,54.79,66.487500
2,03,AlicanteAlacant,M08,M11,M08,M11,159.22,8.13,136.07,11.54,106.207500
3,04,Almera,M08,M09,M08,M09,163.14,16.26,137.15,19.80,89.287500
4,05,vila,M04,M06,M08,M03,75.19,6.75,36.13,30.83,69.874167


## Union de los distintos dataframes a dos niveles: provincia y municipio

El mayor reto al que nos enfrentamos es que, aunque tenemos diferentes identificadores para muchos de los dataframes, no se comparten entre todos, asi que tendremos que encontrar la forma de relacionarlos correctamente

Comenzamos repasando lo que tenemos: Los municipios funcionan con `COD_INE`, valor de 6 dígitos.

Las provincias funcionan con `COD_PROV`, con un valor de dos dígitos (del 01 al 50)

Tenemos además algunos valores por el código EOH, que no tiene una transformación oficial del código INE  a EOH, por lo que se debe aproximar por el nombre

In [89]:
# Observamos que seguimos con el problema de que algunos municipios aparecen con un código interno del INE, debemos corregirlo
INE_puntosTur[INE_puntosTur['COD_INE'].str.len() < 4]

,COD_INE,LOCALIDAD,VALOR PERNOCTACIONES EXTRANJERO,VALOR PERNOCTACIONES NACIONAL,VALOR VIAJERO EXTRANJERO,VALOR VIAJERO NACIONAL,PERIODO PERNOCTACIONES EXTRANJERO,PERIODO PERNOCTACIONES NACIONAL,PERIODO VIAJERO EXTRANJERO,PERIODO VIAJERO NACIONAL
28,B1,Almuñécar,714392.0,860498.0,111035.0,280368.0,M10,M08,M09,M08
29,B4,Antigua,4253440.0,335176.0,586949.0,96519.0,M08,M08,M03,M08
30,H4,"Oliva, La",5361772.0,483732.0,720158.0,109872.0,M10,M08,M10,M08
31,K7,Teguise,4480139.0,681309.0,605654.0,134795.0,M08,M08,M03,M08
32,M4,Yaiza,9065358.0,1276782.0,1222484.0,236113.0,M10,M08,M10,M08
...,...,...,...,...,...,...,...,...,...,...
105,L6,Valladolid,88514.0,320994.0,50952.0,190530.0,M07,M06,M07,M06
106,M1,Vigo,165676.0,311203.0,95080.0,156371.0,M07,M07,M05,M07
107,M5,Zamora,12288.0,90476.0,8395.0,55879.0,M05,M04,M05,M04
108,M6,Ávila,51951.0,197429.0,35636.0,129044.0,M05,M05,M05,M05


In [90]:

# Diccionario para pasar del código EOH a Cod INE
map_eoh_a_ine = {
    'A1': '38001', 'A5': '02003', 'A6': '44009', 'A8': '11004', 'B0': '04013',
    'B1': '18017', 'B4': '35003', 'B7': '38006', 'C2': '29025', 'C4': '48020',
    'C6': '09059', 'D1': '30016', 'D4': '15030', 'D6': '16078', 'D8': '10037',
    'D9': '11012', 'E0': '14021', 'E2': '20069', 'E3': '11027', 'E6': '29054',
    'E7': '46131', 'E8': '33024', 'E9': '18087', 'F4': '11020', 'F5': '35016',
    'F6': '24089', 'F7': '33036', 'F9': '25120', 'G0': '26089', 'G1': '27028',
    'G2': '28079', 'G3': '29069', 'G6': '35012', 'G7': '04064', 'H0': '30030',
    'H1': '29067', 'H2': '06083', 'H3': '29075', 'H4': '35014', 'H6': '33044',
    'H7': '35015', 'H9': '31201', 'I4': '38028', 'I6': '27051', 'I7': '29084',
    'I9': '37274', 'J0': '35019', 'J5': '38038', 'J6': '39075', 'J7': '15078',
    'K0': '40194', 'K2': '41091', 'K3': '42173', 'K4': '11035', 'K5': '43148',
    'K7': '35024', 'K8': '44216', 'L0': '35028', 'L1': '45168', 'L2': '29901',
    'L5': '46250', 'L6': '47186', 'M1': '36057', 'M4': '35034', 'M5': '49275',
    'M6': '05019', 'M7': '50297', 'M9': '11014', 'Q4': '13034', 'Q5': '17079',
    'Q7': '19257', 'R2': '37107'
}


# Reemplazar los códigos cortos por los códigos INE
INE_puntosTur['COD_INE'] = INE_puntosTur['COD_INE'].apply(lambda x: map_eoh_a_ine.get(x, x))

#Revisamos el largo (nº de caracteres) diferentes, si esta correcto deberian ser solo 5
INE_puntosTur['COD_INE'].str.len().value_counts()


COD_INE
5    108
Name: count, dtype: int64

Con los codigos corregidos, realizamos el merge en los datos de nivel municipio

In [91]:
print(f'municipiosDF: {municipiosDF.shape}')
print(f'municipios_data: {municipios_data.shape}')
print(f'INE_puntosTur: {INE_puntosTur.shape}')

municipiosDF: (8132, 16)
municipios_data: (8132, 6)
INE_puntosTur: (108, 10)


In [92]:
# COD_INE, en municipios_data debe ser un int64 para cruzarlo con municipiosDF
municipios_data['COD_INE'] = municipios_data['COD_INE'].astype('Int64')

#formateamos el valor de COD_INE para cruzarlo correctamente
municipiosDF['COD_INE'] = municipiosDF['COD_INE'].astype(str).str[:-6].astype('Int64')

municipiosDF = municipiosDF.merge(propias.añadir_prefijo_col(municipios_data, 
                                                     'COD_INE', 
                                                     'Personas_'),
                                  left_on='COD_INE',
                                  right_on= 'COD_INE')


print(f'municipiosDF: {municipiosDF.shape}')
print(f'municipios_data: {municipios_data.shape}')
print(f'INE_puntosTur: {INE_puntosTur.shape}')


municipiosDF: (8132, 21)
municipios_data: (8132, 6)
INE_puntosTur: (108, 10)


In [93]:
print('Estudiamos la forma de los Dfs: ')
print('----------------------------------')
print(f'provinciasDF: {provinciasDF.shape}')
print(f'f_INE_provincias: {f_INE_provincias.shape}')
print(f'rentabilidad_H: {rentabilidad_H.shape}')
print(f'metricas_provincia: {metricas_provincia.shape}')

Estudiamos la forma de los Dfs: 
----------------------------------
provinciasDF: (52, 5)
f_INE_provincias: (50, 6)
rentabilidad_H: (52, 11)
metricas_provincia: (52, 10)


Como un DF tiene 50 filas en vez de 52, debemos realizar en ``f_INE_provincias`` un left merge

In [94]:
provinciasDF = provinciasDF.merge(propias.añadir_prefijo_col(f_INE_provincias, 
                                                     'COD_PROV', 
                                                     'Perc_'),
                                  how='left',
                                  left_on='COD_PROV',
                                  right_on= 'COD_PROV')

provinciasDF = provinciasDF.merge(propias.añadir_prefijo_col(rentabilidad_H, 
                                                     'COD_INE', 
                                                     'Ratios_'),
                                  left_on='COD_PROV',
                                  right_on= 'COD_INE')

provinciasDF = provinciasDF.merge(propias.añadir_prefijo_col(metricas_provincia, 
                                                     'cod_prov', 
                                                     'Metricas_'),
                                  left_on='COD_PROV',
                                  right_on= 'cod_prov')



print('Comprobamos shape de provinciasDF despues de unir todos los df')
print('---------------------------------------------------------------')
print(f'provinciasDF: {provinciasDF.shape}')

Comprobamos shape de provinciasDF despues de unir todos los df
---------------------------------------------------------------
provinciasDF: (52, 31)


Para finalizar esta parte del proyecto, guardamos los archivos ya trabajados en la carpeta Datos Procesados

In [95]:
# Finalmente, guardamos todos los DataFrames ya revisados y procesados

# Si se ejecuta en Colab
if "google.colab" in sys.modules:
    posibles_drive = [
        Path("/content/drive/MyDrive/TUI AI DASHBOARD CODE"),
        Path("/content/drive/MyDrive/TFM - TUI/TUI AI DASHBOARD CODE"),
        Path("/content/drive/MyDrive/TUI-AI-DASHBOARD-TFM"),
    ]
    raiz = next((p for p in posibles_drive if p.exists()), Path("/content/drive/MyDrive/TUI AI DASHBOARD CODE"))
    carpeta = raiz / 'Datos procesados'
else:
    carpeta = Path('Datos procesados')

#Si se ejecuta en local
carpeta.mkdir(parents=True, exist_ok=True)

# Datos geográficos y puntos turísticos
municipiosDF.to_csv(carpeta / 'MunicipiosGeo.csv', index=False, sep=';', encoding='UTF-8')
provinciasDF.to_csv(carpeta / 'ProvinciasGeo.csv', index=False, sep=';', encoding='UTF-8')
INE_puntosTur.to_csv(carpeta / 'PuntosTuristicos.csv', index=False, sep=';', encoding='UTF-8')
